# 🚦 Detección de Infracciones de Tráfico con YOLOv11

Este notebook es una **demo educativa de extremo a extremo** sobre detección de objetos aplicada al dominio del tráfico vial. Cubre el ciclo completo de un proyecto de visión por computador:

1. **Análisis Exploratorio de Datos (EDA)** — entender el dataset antes de entrenar.
2. **Diseño de la Red Neuronal** — arquitectura YOLOv11 y transfer learning.
3. **Entrenamiento** — configuración de hiperparámetros y bucle de optimización.
4. **Validación** — métricas estándar de detección de objetos (mAP, precision, recall).
5. **Evaluación e Inferencia** — predicciones visuales y análisis de confianza.

---
### ¿Qué es YOLO?
**YOLO** (*You Only Look Once*) es una familia de modelos de detección de objetos en tiempo real. A diferencia de los detectores en dos etapas (como Faster R-CNN), YOLO procesa la imagen completa en una sola pasada por la red, lo que lo hace extremadamente rápido.

**YOLOv11** es la versión más reciente de Ultralytics, con mejoras en precisión y eficiencia respecto a YOLOv8. La variante **nano (`yolo11n`)** es la más ligera: ideal para prototipado y entornos con recursos limitados.

---
### Dataset
Usamos el dataset público [Traffic Violation Detection](https://www.kaggle.com/datasets/guisahanes/traffic-violation-detection-dataset) de Kaggle, que contiene imágenes de tráfico anotadas con bounding boxes en formato YOLO para múltiples tipos de infracciones y vehículos.


### Instalación de dependencias

Instalamos las librerías necesarias: `ultralytics` (YOLO), `kagglehub` (descarga del dataset), `opencv-python` (procesado de imágenes), `matplotlib` y `seaborn` (visualización).

In [ ]:
# Instalación de las librerías necesarias (solo la primera vez o en entornos nuevos).
# --quiet suprime la salida verbosa de pip para mantener el notebook limpio.
#
# Paquetes instalados:
#   - ultralytics : framework oficial de YOLO (entrenamiento, validación, inferencia)
#   - kagglehub   : descarga datasets y modelos directamente desde Kaggle
#   - opencv-python (cv2) : lectura/escritura de imágenes y conversión de espacios de color
#   - matplotlib  : visualización de imágenes, gráficas y curvas de entrenamiento
#   - seaborn     : gráficas estadísticas de alto nivel (importado pero disponible para EDA)
%pip install ultralytics kagglehub opencv-python matplotlib seaborn --quiet

### Importaciones y configuración del dispositivo

Importamos todas las librerías y detectamos automáticamente si hay GPU disponible. Entrenar en GPU puede ser **10-50x más rápido** que en CPU.

In [ ]:
import os                          # Operaciones del sistema de ficheros (rutas, existencia de archivos)
import yaml                        # Lectura/escritura de archivos YAML (configuración del dataset)
import random                      # Selección aleatoria de imágenes de muestra
import numpy as np                 # Operaciones numéricas y arrays
import torch                       # Framework de deep learning; aquí solo para detectar GPU
import cv2                         # OpenCV: lectura de imágenes y conversión BGR→RGB
import matplotlib.pyplot as plt    # Visualización principal
import matplotlib.patches as patches  # Dibujo de rectángulos (bounding boxes) sobre imágenes
import seaborn as sns              # Gráficas estadísticas (disponible para uso adicional)
from pathlib import Path           # Manejo de rutas de forma orientada a objetos
from collections import Counter    # Conteo eficiente de elementos (frecuencia de clases)
import kagglehub                   # Cliente de Kaggle para descargar datasets
from ultralytics import YOLO       # Clase principal del framework Ultralytics

# Selección automática del dispositivo de cómputo:
#   - Si hay GPU disponible (CUDA), usamos la primera (índice 0) → mucho más rápido
#   - Si no hay GPU, usamos CPU → más lento pero funcional
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

---
## 1. 📊 Análisis Exploratorio de Datos (EDA)

Antes de entrenar cualquier modelo es fundamental **conocer el dataset**. El EDA nos permite:

- Verificar que el dataset se ha descargado correctamente y tiene la estructura esperada.
- Conocer el número de clases y sus nombres.
- Detectar **desbalanceo de clases** (algunas clases con muchas más muestras que otras), lo que puede sesgar el modelo.
- Analizar la **distribución de tamaños de bounding boxes**: si los objetos son muy pequeños, puede ser necesario aumentar la resolución de entrada.
- Inspeccionar visualmente las anotaciones para detectar errores de etiquetado.

### Formato de anotaciones YOLO
Cada imagen tiene un archivo `.txt` asociado. Cada línea representa un objeto:
```
<class_id> <cx> <cy> <width> <height>
```
Todos los valores están **normalizados** entre 0 y 1 respecto al tamaño de la imagen. `(cx, cy)` es el centro del bounding box.


In [ ]:
# ── Descarga del dataset desde Kaggle ────────────────────────────────────────
# kagglehub.dataset_download descarga el dataset a la caché local (~/.cache/kagglehub)
# y devuelve la ruta al directorio raíz del dataset.
# Si ya fue descargado previamente, reutiliza la caché sin volver a descargar.
path = kagglehub.dataset_download("guisahanes/traffic-violation-detection-dataset")
print(f"Dataset descargado en: {path}")

# ── Lectura del archivo de configuración data.yaml ───────────────────────────
# data.yaml es el archivo estándar de Ultralytics que describe el dataset:
#   - path  : ruta raíz del dataset
#   - train : ruta relativa a las imágenes de entrenamiento
#   - val   : ruta relativa a las imágenes de validación
#   - nc    : número de clases (number of classes)
#   - names : lista o diccionario con los nombres de las clases
DATA_YAML = f"{path}/data.yaml"
with open(DATA_YAML) as f:
    config = yaml.safe_load(f)  # safe_load evita ejecución de código arbitrario en el YAML

# Los nombres de clase pueden venir como lista ['car', 'bus', ...]
# o como diccionario {0: 'car', 1: 'bus', ...} según la versión del dataset.
# Normalizamos siempre a lista para acceso por índice.
CLASS_NAMES = list(config['names'].values()) if isinstance(config['names'], dict) else config['names']
NUM_CLASSES = config['nc']  # Número total de clases a detectar
print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")

### Conteo de imágenes y etiquetas por split

Verificamos que el número de imágenes y archivos de etiquetas coincide en cada split (`train` / `val`). Una discrepancia indicaría imágenes sin anotar o etiquetas huérfanas.

In [ ]:
# ── Conteo de imágenes y anotaciones por split (train / val) ─────────────────
# Un dataset bien preparado debe tener exactamente una etiqueta por imagen.
# Si hay más imágenes que etiquetas, algunas imágenes no tienen objetos anotados
# (imágenes negativas), lo cual puede ser intencional o un error.

# Rutas relativas dentro del directorio del dataset
splits = {'train': 'images/train', 'val': 'images/val'}
label_splits = {'train': 'labels/train', 'val': 'labels/val'}

stats = {}
for split, img_dir in splits.items():
    img_path = Path(path) / img_dir
    lbl_path = Path(path) / label_splits[split]

    # Recogemos imágenes en formato JPG y PNG
    images = list(img_path.glob('*.jpg')) + list(img_path.glob('*.png'))

    # Las etiquetas son archivos .txt; si la carpeta no existe, asumimos 0 etiquetas
    labels = list(lbl_path.glob('*.txt')) if lbl_path.exists() else []

    stats[split] = {'images': len(images), 'labels': len(labels)}

print("Split\t\tImágenes\tEtiquetas")
for split, s in stats.items():
    print(f"{split}\t\t{s['images']}\t\t{s['labels']}")

### Distribución de clases y tamaños de bounding boxes

Analizamos cuántas instancias hay por clase (para detectar desbalanceo) y la distribución de tamaños de los bounding boxes (para evaluar si los objetos son pequeños o grandes en relación a la imagen).

In [ ]:
# ── Análisis de distribución de clases y tamaños de bounding boxes ───────────
# Leemos TODOS los archivos de etiquetas del conjunto de entrenamiento para:
#   1. Contar cuántas instancias hay de cada clase → detectar desbalanceo
#   2. Recopilar anchos y altos de los bounding boxes → entender escala de objetos

train_labels_path = Path(path) / 'labels/train'
class_counts = Counter()       # Acumula {class_id: total_instances}
bbox_widths, bbox_heights = [], []  # Listas de dimensiones normalizadas [0,1]

for lbl_file in train_labels_path.glob('*.txt'):
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:  # Formato válido: class cx cy w h
                # Desempaquetamos: cls=entero, resto=floats
                cls, _, _, w, h = int(parts[0]), *map(float, parts[1:])
                class_counts[cls] += 1
                bbox_widths.append(w)   # Ancho relativo al tamaño de la imagen
                bbox_heights.append(h)  # Alto relativo al tamaño de la imagen

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Gráfico 1: Barras horizontales con frecuencia de cada clase ───────────────
# Ordenamos por class_id para mantener consistencia con CLASS_NAMES
sorted_classes = sorted(class_counts.items())
names  = [CLASS_NAMES[c] for c, _ in sorted_classes]
counts = [cnt for _, cnt in sorted_classes]
# Paleta de 20 colores distintos para diferenciar clases visualmente
colors = plt.cm.tab20(np.linspace(0, 1, len(names)))
axes[0].barh(names, counts, color=colors)
axes[0].set_title('Distribución de clases (train)')
axes[0].set_xlabel('Número de instancias')
# Un desbalanceo severo (ej. una clase con 10x más instancias) puede requerir
# técnicas como oversampling, class weights o focal loss.

# ── Gráfico 2: Scatter de dimensiones de bounding boxes ──────────────────────
# Cada punto representa un objeto. Valores cercanos a 0 = objetos pequeños.
# Si la mayoría de puntos están en la esquina inferior izquierda, los objetos
# son pequeños y puede ser necesario usar imgsz > 640 o modelos específicos.
axes[1].scatter(bbox_widths, bbox_heights, alpha=0.3, s=5, color='steelblue')
axes[1].set_title('Tamaño de Bounding Boxes (normalizado)')
axes[1].set_xlabel('Ancho relativo')
axes[1].set_ylabel('Alto relativo')

plt.tight_layout()
plt.show()
print(f"Total de instancias en train: {sum(counts)}")

### Visualización de muestras con anotaciones

Inspeccionamos visualmente las primeras imágenes del conjunto de entrenamiento con sus bounding boxes de ground truth superpuestos, para verificar la calidad del etiquetado.

In [ ]:
# ── Visualización de imágenes de muestra con sus bounding boxes ───────────────
# Inspeccionar visualmente las anotaciones es crucial para detectar:
#   - Bounding boxes mal alineados o con clase incorrecta
#   - Objetos sin anotar (falsos negativos en el ground truth)
#   - Calidad general de las imágenes (resolución, iluminación, oclusiones)

def draw_annotations(img_path, lbl_path, class_names):
    """Dibuja los bounding boxes de ground truth sobre una imagen.
    
    Args:
        img_path   : Path a la imagen
        lbl_path   : Path al archivo .txt de etiquetas YOLO
        class_names: Lista de nombres de clase indexada por class_id
    Returns:
        fig: figura de matplotlib con la imagen anotada
    """
    # OpenCV lee en BGR; convertimos a RGB para matplotlib
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]  # Dimensiones reales en píxeles (necesarias para desnormalizar)
    fig, ax = plt.subplots(1, figsize=(6, 4))
    ax.imshow(img)
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                # Desnormalizamos las coordenadas YOLO a píxeles absolutos
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                # YOLO usa centro (cx,cy); matplotlib usa esquina superior izquierda (x1,y1)
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor='lime', facecolor='none'  # Borde verde, sin relleno
                )
                ax.add_patch(rect)
                # Etiqueta de clase sobre el bounding box con fondo semitransparente
                ax.text(x1, y1 - 4, class_names[int(cls)], color='lime', fontsize=7,
                        bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.axis('off')
    ax.set_title(img_path.name, fontsize=8)
    return fig

# Tomamos las primeras 6 imágenes del conjunto de entrenamiento como muestra
train_imgs = list((Path(path) / 'images/train').glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, img_p in zip(axes.flat, train_imgs):
    # El archivo de etiquetas tiene el mismo nombre que la imagen pero extensión .txt
    lbl_p = Path(path) / 'labels/train' / (img_p.stem + '.txt')
    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    if lbl_p.exists():
        with open(lbl_p) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                # Conversión de coordenadas normalizadas a píxeles
                x1, y1 = (cx - bw/2)*w, (cy - bh/2)*h
                rect = patches.Rectangle(
                    (x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor='lime', facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1-4, CLASS_NAMES[int(cls)], color='lime', fontsize=6,
                        bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.axis('off')
    ax.set_title(img_p.name, fontsize=7)

plt.suptitle('Muestras del conjunto de entrenamiento con anotaciones', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. 🧠 Diseño de la Red Neuronal

### Arquitectura YOLOv11n

Usamos **YOLOv11n** (nano), el modelo más ligero de la familia YOLO11. Su arquitectura sigue el patrón clásico de los detectores modernos de una sola etapa:

| Componente | Módulos | Descripción |
|---|---|---|
| **Backbone** | C3k2 + SPPF | Extrae mapas de características a múltiples escalas. C3k2 es una versión eficiente de CSP; SPPF (Spatial Pyramid Pooling Fast) captura contexto global. |
| **Neck (PAN-FPN)** | C2PSA | Fusiona características de distintas resoluciones (Feature Pyramid Network + Path Aggregation Network). Permite detectar objetos grandes y pequeños simultáneamente. |
| **Head** | Detect | Para cada celda de la cuadrícula predice: coordenadas del bounding box (x,y,w,h), puntuación de objectness y probabilidades de clase. |

### Transfer Learning

En lugar de entrenar desde cero (*from scratch*), partimos de **pesos preentrenados en COCO** (dataset con 80 clases y ~330k imágenes). Esto nos proporciona:

- **Convergencia más rápida**: el backbone ya sabe detectar bordes, texturas y formas.
- **Mejor generalización** con menos datos de entrenamiento.
- **Menor riesgo de overfitting**.

Al entrenar con nuestro dataset, la cabeza de detección se adapta a las **23 clases** específicas de infracciones de tráfico, mientras el backbone se afina (*fine-tuning*) con un learning rate bajo.

### Función de pérdida
YOLOv11 usa tres componentes de pérdida:
- **Box loss** (DFL + CIoU): penaliza la imprecisión en las coordenadas del bounding box.
- **Class loss** (BCE): penaliza la clasificación incorrecta del objeto.
- **DFL loss** (*Distribution Focal Loss*): mejora la precisión de los bordes del bounding box modelando la distribución de probabilidad de las coordenadas.


In [ ]:
# ── Carga del modelo base preentrenado ───────────────────────────────────────
# YOLO('yolo11n.pt') descarga automáticamente los pesos preentrenados en COCO
# desde los servidores de Ultralytics si no están en caché local.
# Este modelo tiene 80 clases de salida (las de COCO).
# Al entrenar con nuestro dataset, la cabeza se reemplaza automáticamente
# para adaptarse a NUM_CLASSES clases.
base_model = YOLO('yolo11n.pt')

# .info() imprime un resumen de la arquitectura:
#   - Número de capas y parámetros totales
#   - Parámetros entrenables vs. congelados
#   - GFLOPs (operaciones de punto flotante): indica el coste computacional
# verbose=False muestra solo el resumen compacto
print("Arquitectura del modelo:")
base_model.info(verbose=False)

### Configuración del archivo data.yaml

El `data.yaml` original puede contener rutas relativas que no funcionan desde la caché de `kagglehub`. Generamos un `data_fixed.yaml` con la ruta absoluta correcta y definimos las constantes del experimento.

In [ ]:
# ── Preparación del data.yaml con rutas absolutas ────────────────────────────
# El data.yaml original del dataset puede contener rutas relativas que no
# funcionan correctamente cuando el dataset está en la caché de kagglehub.
# Solución: sobreescribir el campo 'path' con la ruta absoluta real y guardar
# un nuevo archivo data_fixed.yaml que usaremos en entrenamiento y validación.

PROJECT_DIR = '.'           # Directorio donde se guardarán los resultados del entrenamiento
MODEL_NAME  = 'traffic_yolo11n'  # Nombre del experimento (crea subcarpeta con este nombre)
# Ruta esperada del mejor modelo tras el entrenamiento
# YOLO guarda automáticamente best.pt (mejor época) y last.pt (última época)
MODEL_PATH  = f'{PROJECT_DIR}/{MODEL_NAME}/weights/best.pt'

# Actualizamos el campo 'path' del config con la ruta absoluta del dataset descargado
fixed_yaml = os.path.join(PROJECT_DIR, 'data_fixed.yaml')
config['path'] = path  # 'path' es la variable con la ruta de kagglehub

# default_flow_style=False genera YAML en formato legible (bloque), no en línea
with open(fixed_yaml, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"data.yaml corregido guardado en: {fixed_yaml}")
print(f"Clases: {NUM_CLASSES} | Dispositivo: {DEVICE}")

---
## 3. 🏋️ Entrenamiento

### Hiperparámetros clave

| Parámetro | Valor | Justificación |
|---|---|---|
| `epochs` | 50 | Suficiente para convergencia con transfer learning; más épocas pueden causar overfitting |
| `imgsz` | 640 | Resolución estándar YOLO: balance óptimo entre velocidad y precisión |
| `batch` | 16 | Adecuado para GPU de 8 GB VRAM; reducir a 8 si aparece error OOM (Out Of Memory) |
| `lr0` | 0.01 | Learning rate inicial; el scheduler coseno lo reduce gradualmente hasta ~0 |
| `cos_lr` | True | Scheduler de coseno: decaimiento suave que evita oscilaciones al final del entrenamiento |
| `warmup_epochs` | 3 | Las primeras 3 épocas el LR sube linealmente desde 0 → evita inestabilidad inicial |
| `weight_decay` | 0.0005 | Regularización L2: penaliza pesos grandes para reducir overfitting |
| `momentum` | 0.937 | Parámetro del optimizador SGD: suaviza las actualizaciones de gradiente |
| `optimizer` | auto | Ultralytics elige automáticamente SGD o AdamW según el tamaño del modelo |
| `pretrained` | True | Usa los pesos preentrenados en COCO como punto de partida (transfer learning) |
| `seed` | 42 | Semilla para reproducibilidad de resultados |

### Data Augmentation
Ultralytics aplica automáticamente augmentaciones durante el entrenamiento:
- **Mosaic**: combina 4 imágenes en una → el modelo ve más contexto y variedad por batch.
- **Random flip horizontal/vertical**: aumenta la invarianza a la orientación.
- **HSV jitter**: variaciones de tono, saturación y brillo → robustez a condiciones de iluminación.
- **Scale/translate/shear**: deformaciones geométricas → robustez a perspectiva.

### Checkpoint y reanudación
La celda siguiente comprueba si ya existe un modelo entrenado (`best.pt`). Si existe, lo carga directamente sin reentrenar, lo que permite **reanudar el trabajo** sin perder tiempo.


In [ ]:
# ── Entrenamiento con checkpoint automático ───────────────────────────────────
# Si ya existe best.pt de una ejecución anterior, lo cargamos directamente.
# Esto evita reentrenar innecesariamente al re-ejecutar el notebook.
if os.path.exists(MODEL_PATH):
    print(f"Modelo ya entrenado encontrado en: {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
else:
    print("Iniciando entrenamiento...")
    # Cargamos el modelo base preentrenado en COCO
    model = YOLO('yolo11n.pt')

    train_result = model.train(
        data=fixed_yaml,       # Ruta al data.yaml con rutas absolutas corregidas
        epochs=50,             # Número total de épocas de entrenamiento
        imgsz=640,             # Tamaño de entrada: las imágenes se redimensionan a 640x640
        batch=16,              # Imágenes por batch; ajustar según VRAM disponible
        name=MODEL_NAME,       # Nombre del experimento → crea carpeta ./traffic_yolo11n/
        project=PROJECT_DIR,   # Directorio raíz donde se guardan los resultados
        device=DEVICE,         # 0=GPU, 'cpu'=CPU
        workers=4,             # Hilos de carga de datos en paralelo (DataLoader workers)
        pretrained=True,       # Inicializa con pesos COCO (transfer learning)
        optimizer='auto',      # Ultralytics elige SGD o AdamW automáticamente
        lr0=0.01,              # Learning rate inicial
        cos_lr=True,           # Scheduler coseno: LR decae suavemente hasta ~0
        warmup_epochs=3,       # Épocas de calentamiento: LR sube linealmente de 0 a lr0
        weight_decay=0.0005,   # Regularización L2 para evitar overfitting
        momentum=0.937,        # Momentum del optimizador SGD
        seed=42,               # Semilla aleatoria para reproducibilidad
        verbose=True           # Muestra métricas por época en consola
    )

    # YOLO puede crear subcarpetas numeradas (traffic_yolo11n2, traffic_yolo11n3...)
    # si el nombre ya existe. Usamos save_dir del resultado para obtener la ruta real.
    MODEL_PATH = str(Path(train_result.save_dir) / 'weights' / 'best.pt')
    model = YOLO(MODEL_PATH)
    print(f"Entrenamiento completado. Modelo cargado desde: {MODEL_PATH}")

print(f"Modelo listo: {MODEL_PATH}")

### Curvas de entrenamiento

Visualizamos la evolución de las pérdidas (`box`, `cls`, `dfl`) y las métricas (`precision`, `recall`, `mAP@0.5`) a lo largo de las épocas. Un gap creciente entre train y val indica overfitting.

In [ ]:
# ── Visualización de curvas de entrenamiento ──────────────────────────────────
# Ultralytics guarda automáticamente un archivo results.csv con las métricas
# de cada época. Analizarlo nos permite diagnosticar el entrenamiento:
#
#   - Box/Class/DFL loss decrecientes → el modelo está aprendiendo
#   - Gap entre train loss y val loss → indica overfitting si es grande
#   - Precision/Recall/mAP crecientes → mejora en detección
#   - Plateau en mAP → el modelo ha convergido

# results.csv está en la carpeta del experimento (un nivel arriba de weights/)
results_csv = Path(MODEL_PATH).parent.parent / 'results.csv'

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Elimina espacios en los nombres de columna

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    # Pares (columna_train, columna_val, título) para cada subgráfica
    # Las métricas de validación (val_col=None) solo tienen una curva
    metrics = [
        ('train/box_loss',          'val/box_loss',  'Box Loss'),
        ('train/cls_loss',          'val/cls_loss',  'Class Loss'),
        ('train/dfl_loss',          'val/dfl_loss',  'DFL Loss'),
        ('metrics/precision(B)',    None,            'Precision'),
        ('metrics/recall(B)',       None,            'Recall'),
        ('metrics/mAP50(B)',        None,            'mAP@0.5'),
    ]

    for ax, (train_col, val_col, title) in zip(axes.flat, metrics):
        # Comprobamos que la columna existe antes de graficar (robustez)
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], label='train')
        if val_col and val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], label='val')
        ax.set_title(title)
        ax.set_xlabel('Época')
        ax.legend()
        ax.grid(True, alpha=0.3)  # Cuadrícula suave para facilitar la lectura

    plt.suptitle('Curvas de Entrenamiento', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("results.csv no encontrado. Ejecuta el entrenamiento primero.")

---
## 4. ✅ Validación

Evaluamos el modelo sobre el **conjunto de validación** (imágenes no vistas durante el entrenamiento) para medir su capacidad de generalización.

### Métricas de detección de objetos

| Métrica | Descripción |
|---|---|
| **Precision** | De todas las detecciones realizadas, ¿qué fracción son correctas? `TP / (TP + FP)` |
| **Recall** | De todos los objetos reales, ¿qué fracción detectamos? `TP / (TP + FN)` |
| **AP@0.5** | Área bajo la curva Precision-Recall con umbral IoU=0.5. Métrica principal por clase. |
| **mAP@0.5** | Media del AP@0.5 sobre todas las clases. Métrica global estándar. |
| **mAP@0.5:0.95** | Media del AP calculado con umbrales IoU de 0.5 a 0.95 (paso 0.05). Más exigente. |

### ¿Qué es IoU?
**Intersection over Union** mide el solapamiento entre el bounding box predicho y el ground truth:
```
IoU = Área(intersección) / Área(unión)
```
Un IoU ≥ 0.5 se considera una detección correcta (True Positive) en el estándar PASCAL VOC.

### Parámetros de validación
- `conf=0.25`: umbral de confianza mínima para considerar una detección válida.
- `iou=0.5`: umbral de IoU para NMS (Non-Maximum Suppression) y para clasificar TP/FP.


In [ ]:
# ── Evaluación sobre el conjunto de validación ────────────────────────────────
# model.val() ejecuta una pasada completa sobre todas las imágenes de validación
# y calcula las métricas estándar de detección de objetos.
val_results = model.val(
    data=fixed_yaml,  # Dataset con rutas absolutas
    imgsz=640,        # Misma resolución que en entrenamiento (importante para consistencia)
    conf=0.25,        # Umbral de confianza: detecciones con score < 0.25 se descartan
    iou=0.5,          # Umbral IoU para NMS: elimina detecciones duplicadas solapadas
    device=DEVICE,
    verbose=True      # Muestra métricas por clase en consola
)

# ── Métricas globales del modelo ──────────────────────────────────────────────
# val_results.box contiene todas las métricas de detección de bounding boxes:
#   .map50    : mAP con IoU=0.5 (métrica principal, más usada en papers)
#   .map      : mAP promediado sobre IoU=[0.5, 0.55, ..., 0.95] (más estricto)
#   .mp       : mean Precision (promedio sobre todas las clases)
#   .mr       : mean Recall (promedio sobre todas las clases)
print(f"\nmAP@0.5:      {val_results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"Precision:    {val_results.box.mp:.4f}")
print(f"Recall:       {val_results.box.mr:.4f}")

### Average Precision por clase

Desglosamos el `mAP@0.5` por clase para identificar qué clases aprende bien el modelo y cuáles son problemáticas (pocas muestras, alta similitud visual con otras clases, etc.).

In [ ]:
# ── Average Precision por clase ───────────────────────────────────────────────
# Analizar el AP por clase es fundamental para identificar:
#   - Clases bien aprendidas (AP alto) vs. clases problemáticas (AP bajo)
#   - Si las clases con pocas muestras tienen peor rendimiento (efecto del desbalanceo)
#   - Qué clases necesitan más datos o augmentación específica

if hasattr(val_results.box, 'ap_class_index') and val_results.box.ap_class_index is not None:
    ap_per_class   = val_results.box.ap50          # Array con AP@0.5 de cada clase
    class_indices  = val_results.box.ap_class_index  # Índices de las clases evaluadas
    class_labels   = [CLASS_NAMES[i] for i in class_indices]  # Nombres legibles

    # Ordenamos de menor a mayor AP para que las clases más difíciles aparezcan arriba
    sorted_idx = np.argsort(ap_per_class)

    fig, ax = plt.subplots(figsize=(10, 7))
    # Gradiente de color: rojo (AP bajo) → verde (AP alto) para identificar rápidamente
    bars = ax.barh(
        [class_labels[i] for i in sorted_idx],
        [ap_per_class[i] for i in sorted_idx],
        color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_idx)))
    )
    # Línea vertical con el mAP global como referencia
    ax.axvline(
        x=val_results.box.map50, color='navy', linestyle='--',
        label=f'mAP@0.5 media: {val_results.box.map50:.3f}'
    )
    ax.set_xlabel('AP@0.5')
    ax.set_title('Average Precision por clase')
    ax.legend()
    ax.set_xlim(0, 1)  # AP siempre entre 0 y 1
    plt.tight_layout()
    plt.show()

### Matriz de confusión

La matriz de confusión muestra, para cada clase real, cómo la clasifica el modelo. Permite detectar confusiones sistemáticas entre clases similares y clases con muchos falsos negativos o positivos.

In [ ]:
# ── Visualización de la matriz de confusión ───────────────────────────────────
# La matriz de confusión muestra, para cada clase real (filas),
# cómo las clasifica el modelo (columnas). Permite identificar:
#   - Confusiones frecuentes entre clases similares (ej. coche vs. furgoneta)
#   - Clases con muchos falsos negativos (objetos no detectados)
#   - Clases con muchos falsos positivos (detecciones incorrectas)
#
# Ultralytics genera automáticamente este plot durante la validación
# y lo guarda en el directorio de resultados.

val_save_dir = Path(val_results.save_dir)

# Preferimos la versión normalizada (valores entre 0 y 1) para comparar clases
# con distinto número de instancias. Si no existe, usamos la versión absoluta.
conf_matrix_path = val_save_dir / 'confusion_matrix_normalized.png'
if not conf_matrix_path.exists():
    conf_matrix_path = val_save_dir / 'confusion_matrix.png'

if conf_matrix_path.exists():
    # Leemos la imagen generada por Ultralytics y la mostramos con matplotlib
    img = cv2.cvtColor(cv2.imread(str(conf_matrix_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Matriz de Confusión (Validación)')
    plt.show()
else:
    print(f"Matriz de confusión no encontrada en: {val_save_dir}")

---
## 5. 🔍 Evaluación e Inferencia

En esta sección ejecutamos el modelo en modo **inferencia** sobre imágenes individuales para:

1. **Visualizar predicciones** con bounding boxes y etiquetas de clase.
2. **Analizar la distribución de confianza** por clase: ¿el modelo es seguro en sus predicciones?

### Diferencia entre validación e inferencia
- **Validación** (`model.val`): compara predicciones con ground truth → calcula métricas cuantitativas.
- **Inferencia** (`model(source=...)`): solo predice, sin comparar con etiquetas → evaluación cualitativa visual.

### Parámetros de inferencia
- `conf=0.25`: umbral de confianza. Subir este valor → menos detecciones pero más precisas. Bajarlo → más detecciones pero más falsos positivos.
- `iou=0.5`: umbral para NMS (*Non-Maximum Suppression*). NMS elimina bounding boxes duplicados que detectan el mismo objeto; si dos boxes se solapan más del 50%, se queda solo el de mayor confianza.
- `stream=True`: modo generador para procesar datasets grandes sin cargar todo en memoria.


In [ ]:
val_images_path = Path(path) / 'images/val'
sample_images = random.sample(
    list(val_images_path.glob('*.jpg')),
    min(6, len(list(val_images_path.glob('*.jpg'))))
)

results = model(
    source=sample_images, conf=0.25, iou=0.5, imgsz=640, device=DEVICE, verbose=False
)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_p, result in zip(axes.flat, sample_images, results):
    img = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)

    # Predicciones en verde
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none'
        ))
        ax.text(x1, y1-4, f"{CLASS_NAMES[int(box.cls)]} {box.conf.item():.2f}",
                color='lime', fontsize=6, bbox=dict(facecolor='black', alpha=0.4, pad=1))

    # Ground truth en rojo — usamos img_p (Path conocido) en lugar de result.path
    lbl_p = Path(path) / 'labels/val' / (img_p.stem + '.txt')
    if lbl_p.exists():
        with open(lbl_p) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1, y1 = (cx - bw/2)*w, (cy - bh/2)*h
                ax.add_patch(patches.Rectangle(
                    (x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
                ))
                ax.text(x1, y1-4, CLASS_NAMES[int(cls)], color='red', fontsize=6,
                        bbox=dict(facecolor='black', alpha=0.4, pad=1))

    ax.set_title(f"{img_p.name} — {len(result.boxes)} pred", fontsize=8)
    ax.axis('off')

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0],[0], color='lime', linewidth=2, label='Predicción'),
    Line2D([0],[0], color='red',  linewidth=2, linestyle='--', label='Ground Truth'),
], loc='lower center', ncol=2, fontsize=10)
plt.suptitle('Predicciones (verde) vs. Ground Truth (rojo)', fontsize=12)
plt.tight_layout()
plt.show()


### Análisis de confianza por clase

Ejecutamos inferencia sobre todo el conjunto de validación con umbral bajo (`conf=0.1`) para analizar la distribución de confianza por clase. Una confianza alta y concentrada indica que el modelo distingue bien esa clase.

In [ ]:
# ── Análisis de distribución de confianza por clase ───────────────────────────
# Ejecutamos inferencia sobre TODO el conjunto de validación con un umbral bajo
# (conf=0.1) para capturar también detecciones de baja confianza.
# Esto nos permite analizar cómo de seguro es el modelo para cada clase:
#   - Confianza alta y concentrada → el modelo distingue bien esa clase
#   - Confianza baja o dispersa → el modelo tiene dudas → puede necesitar más datos

all_results = model(
    source=str(val_images_path),  # Directorio completo de validación
    conf=0.1,    # Umbral bajo para capturar predicciones inciertas también
    iou=0.5,
    imgsz=640,
    device=DEVICE,
    verbose=False,
    stream=True  # Modo generador: procesa imagen a imagen sin cargar todo en RAM
)

# Acumulamos las puntuaciones de confianza de cada detección, agrupadas por clase
conf_by_class = {name: [] for name in CLASS_NAMES}  # Dict: clase → lista de confianzas

for r in all_results:          # Iteramos sobre cada imagen (generador)
    for box in r.boxes:        # Iteramos sobre cada detección en la imagen
        cls_idx = int(box.cls.item())              # Índice de clase (tensor → int)
        conf_by_class[CLASS_NAMES[cls_idx]].append(box.conf.item())  # Confianza (tensor → float)

# Filtramos clases sin ninguna detección (no aparecen en el boxplot)
detected = {k: v for k, v in conf_by_class.items() if v}

# ── Boxplot de confianza por clase ────────────────────────────────────────────
# Cada caja muestra: mediana, cuartiles Q1/Q3, bigotes y outliers.
# Interpretación:
#   - Caja alta (mediana > 0.7) → el modelo es seguro para esa clase
#   - Caja baja o con muchos outliers → el modelo es inseguro
fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot(detected.values(), labels=detected.keys(), vert=True)

# Línea de referencia en conf=0.25 (umbral estándar de inferencia)
ax.axhline(y=0.25, color='red', linestyle='--', label='Umbral conf=0.25')
ax.set_ylabel('Confianza')
ax.set_title('Distribución de confianza por clase detectada')
ax.legend()
plt.xticks(rotation=45, ha='right')  # Rotamos etiquetas para evitar solapamiento
plt.tight_layout()
plt.show()

### Resumen final

Resumen compacto de las métricas globales del modelo entrenado.

In [ ]:
# ── Resumen final de métricas del modelo ──────────────────────────────────────
# Imprimimos un resumen compacto con las métricas más relevantes para
# tener una visión rápida del rendimiento del modelo entrenado.
#
# Guía de interpretación:
#   mAP@0.5 > 0.5  → rendimiento aceptable para un dataset de 23 clases
#   mAP@0.5 > 0.7  → buen rendimiento
#   mAP@0.5 > 0.9  → rendimiento excelente
#
# mAP@0.5:0.95 siempre será menor que mAP@0.5 porque exige mayor precisión
# en la localización del bounding box.
print("=" * 45)
print("       RESUMEN FINAL DEL MODELO")
print("=" * 45)
print(f"  Modelo:        YOLOv11n (transfer learning)")
print(f"  Clases:        {NUM_CLASSES}")
print(f"  mAP@0.5:       {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95:  {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")
print(f"  Dispositivo:   {DEVICE}")
print("=" * 45)